In [ ]:
# Install the libraries this notebook depends on:
# - nltk: splits text into sentences (used by semantic chunking)
# - sentence-transformers: turns text into embeddings (dense vectors)
# - scikit-learn: gives us cosine_similarity for comparing embeddings
!pip install nltk sentence-transformers scikit-learn

## Document Side Strategies - Chunking Techniques

In [ ]:
import re
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Sample input text
# A mock loan-policy document, deliberately split into 4 clearly labeled
# sections. We'll use this same text to compare how each chunking strategy
# behaves differently on identical input.
sample_text = """
Section 1: Loan Eligibility
To qualify for a personal loan, the customer must be between 21 and 60 years old, have a minimum salary of ₹25,000, and a credit score above 700.

Section 2: Disbursal and Timelines
Loan disbursal typically occurs within 48 hours of approval. Delays can occur due to incomplete documentation.

Section 3: EMI Defaults
If the customer misses 2 or more EMIs, penalties apply. Further defaults may lead to legal action or freezing of accounts.

Section 4: Foreclosure
Customers can foreclose their loan after 6 EMIs have been paid. A foreclosure fee of 2% is applicable.
"""

def display_chunks(chunks, title):
    # Small utility to print out any list of chunks in a readable format,
    # so we can eyeball how each strategy divided up the document.
    print(f"\n{title} -- Total Chunks: {len(chunks)}")
    for i, chunk in enumerate(chunks):
        print(f"\n--- Chunk {i+1} ---\n{chunk}")

def fixed_size_chunking(text, chunk_size=300, overlap=50):
    """
    Split text into fixed-size chunks with optional overlap (measured in tokens).

    This is the simplest possible strategy: just count words and cut.
    It has NO awareness of sentence or section boundaries, so it can
    (and often does) slice a policy rule right down the middle -- e.g.
    splitting "customers must have a credit score above 700" across two
    separate chunks. The `overlap` parameter exists specifically to reduce
    that risk: each new chunk repeats the last `overlap` tokens of the
    previous one, so a detail that falls near a cut point still has a
    chance to appear fully in at least one chunk.
    """
    tokens = text.split()
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk = ' '.join(tokens[start:end])
        chunks.append(chunk)
        # Advance by (chunk_size - overlap) instead of chunk_size, so
        # consecutive chunks share `overlap` tokens with each other.
        start += chunk_size - overlap
    return chunks

# Using a small chunk_size (40) and overlap (10) here purely so the demo
# output is short and easy to read -- in production you'd typically use
# a much larger chunk_size (e.g. 300) tuned to your embedding model's
# context window.
fixed_chunks = fixed_size_chunking(sample_text, chunk_size=40, overlap=10)
display_chunks(fixed_chunks, "Fixed-Size Chunking")

def hierarchical_chunking(text):
    """
    Split document using structured section headers (e.g., "Section 1:", "Section 2:").

    Unlike fixed-size chunking, this respects the document's own structure.
    Because our sample text is consistently formatted with "Section N:"
    headers, splitting right before each header guarantees that every
    chunk is a complete, self-contained section -- no rule gets cut in half.
    The trade-off: this only works because the document happens to be
    well-structured. A messier document (inconsistent headers, no headers
    at all) would need a different or smarter parsing rule.
    """
    # The regex uses a lookahead (?=...) so the "Section N:" text stays
    # attached to the section that follows it, rather than being consumed
    # by the split itself.
    sections = re.split(r'\n(?=Section \d+:)', text.strip())
    return [sec.strip() for sec in sections if sec.strip()]

hierarchical_chunks = hierarchical_chunking(sample_text)
display_chunks(hierarchical_chunks, "Hierarchical Chunking")

import nltk
# Downloads the sentence-tokenizer data nltk needs for sent_tokenize()
# to correctly split text into individual sentences.
nltk.download('punkt_tab')

def semantic_chunking(text, threshold=0.75):
    """
    Group nearby sentences based on semantic similarity (using cosine similarity).

    This is the most content-aware strategy. Instead of relying on document
    structure (like hierarchical chunking) or raw word counts (like
    fixed-size chunking), it looks at *meaning*:
      1. Split the text into individual sentences.
      2. Embed every sentence into a vector using a pretrained encoder.
      3. Walk through sentences in order, comparing each one to the
         previous one via cosine similarity.
      4. As long as consecutive sentences stay "on topic" (similarity >=
         threshold), keep them in the same chunk. Once the topic shifts
         (similarity drops below threshold), start a new chunk.

    Chunks form naturally around topic boundaries rather than arbitrary
    token counts or headers -- which is exactly what protects against the
    "silent failure" described in the PDF, where a naive split cuts a
    policy rule in two and the retriever ends up returning an incomplete,
    misleading answer.
    """
    # Load the pretrained sentence-embedding model. All-MiniLM-L6-v2 is a
    # small, fast model that's a common default choice for this kind of task.
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = sent_tokenize(text)
    embeddings = model.encode(sentences)

    print(sentences)

    chunks = []
    current_chunk = [sentences[0]]
    for i in range(1, len(sentences)):
        # Compare this sentence to the immediately preceding one -- not to
        # the whole current chunk -- so the check reflects a local topic
        # shift rather than an average that could mask an abrupt change.
        sim = cosine_similarity([embeddings[i]], [embeddings[i-1]])[0][0]
        if sim >= threshold:
            # Similar enough to the previous sentence -> same topic,
            # keep building the current chunk.
            current_chunk.append(sentences[i])
        else:
            # Topic shifted -> close off the current chunk and start a new one.
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentences[i]]
    if current_chunk:
        # Don't forget to flush whatever's left in progress at the end.
        chunks.append(' '.join(current_chunk))
    return chunks

semantic_chunks = semantic_chunking(sample_text, threshold=0.75)
display_chunks(semantic_chunks, "Semantic Chunking")

## Retreival side Strategies - Top K, MMR and Reranker

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# A small pool of retrieval candidates. Note the deliberate overlap in
# meaning between chunks -- e.g. two chunks both mention "21 years old",
# and two both mention "credit score of 700". This mirrors the redundancy
# problem the PDF calls out: real document collections often have
# multiple chunks saying almost the same thing, and vanilla similarity
# search has no built-in way to avoid picking several of them at once.
chunks = [
    "Applicants must be at least 21 years old.",
    "Minimum monthly salary is ₹25,000.",
    "Applicants must have a credit score above 700.",
    "Applicants must be at least 21 years old and earn ₹25,000.",
    "Minimum credit score of 700 is required.",
    "Personal loans are not available to self-employed applicants.",
    "EMI defaults attract a penalty of 2% per month.",
]

query = "What are the age, salary, and credit score requirements for a personal loan?"

# IMPORTANT: the query and the chunks are embedded with the SAME model.
# The PDF stresses this point heavily -- if the query and documents were
# embedded with different/mismatched models, the resulting vectors would
# live in different "coordinate systems" and cosine similarity between
# them would be meaningless, even though it would still return a number.
model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = model.encode(chunks)
query_embedding = model.encode([query])[0]

def vanilla_top_k(chunk_embeddings, query_embedding, k=3):
    """
    Return the top-k chunks purely by cosine similarity to the query.

    This is the "naive" baseline retrieval strategy: rank every chunk by
    how similar it is to the query, and return the k highest-scoring ones.
    It has no concept of diversity, so if several chunks express nearly
    the same fact, vanilla top-k is happy to return all of them --
    potentially at the expense of a different, less "loud" but equally
    important fact.
    """
    similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]
    # argsort() sorts ascending; take the last k and reverse them so the
    # most similar chunk comes first.
    top_k_idx = similarities.argsort()[-k:][::-1]
    return [chunks[i] for i in top_k_idx], similarities[top_k_idx]

# k=7 here so we can see the FULL ranking of every chunk (there are only
# 7 total), which makes it easy to compare against MMR's selection below.
vanilla_results, vanilla_scores = vanilla_top_k(chunk_embeddings, query_embedding, k=7)
print("\nVanilla Top-3 Results:")
for text, score in zip(vanilla_results, vanilla_scores):
    print(f"  - {text}  [score: {score:.2f}]")

def mmr(chunk_embeddings, query_embedding, k=3, lambda_param=0.7):
    """
    Returns k chunks using Maximum Marginal Relevance (MMR)

    MMR directly addresses the redundancy problem: instead of picking the
    k most relevant chunks independently, it picks them one at a time and,
    after the first pick, penalizes any candidate that looks too similar
    to what's already been selected. That way each new chunk has to earn
    its place by adding NEW information, not just repeating a popular fact.

    lambda_param controls the relevance/diversity trade-off:
      - closer to 1 -> prioritize pure relevance to the query
      - closer to 0 -> prioritize novelty/diversity between chunks
    """
    # Calculate cosine similarities
    similarities = cosine_similarity(chunk_embeddings, [query_embedding]).flatten()
    selected = []
    selected_idx = []
    candidate_idx = list(range(len(chunks)))

    for _ in range(k):
        if len(selected_idx) == 0:
            # First pick is always just the single most relevant chunk --
            # there's nothing yet to compare it against for diversity.
            idx = np.argmax(similarities)
            selected_idx.append(idx)
            selected.append(chunks[idx])
            candidate_idx.remove(idx)
            continue
        # For each remaining candidate, find how similar it is to the
        # MOST similar chunk already selected. This is the "diversity
        # penalty" -- a candidate that echoes anything already picked
        # gets penalized, even if it's a great match for the query.
        diversity = np.array([
            max(cosine_similarity([chunk_embeddings[i]], [chunk_embeddings[j]])[0][0] for j in selected_idx)
            for i in candidate_idx
        ])
        # Balance relevance (similarity to the query) against diversity
        # (similarity to what's already chosen) using lambda_param.
        mmr_score = lambda_param * similarities[candidate_idx] - (1 - lambda_param) * diversity
        idx = candidate_idx[np.argmax(mmr_score)]
        selected_idx.append(idx)
        selected.append(chunks[idx])
        candidate_idx.remove(idx)
    return selected

mmr_results = mmr(chunk_embeddings, query_embedding, k=3, lambda_param=0.7)
print("\nMMR Top-3 Results:")
for text in mmr_results:
    print(f"  - {text}")

## Rerankers

In [ ]:
# Same chunk texts as before, but now each one carries metadata:
# - "year": when this policy text was written/updated
# - "type": whether it's an official "policy" line or a "faq" snippet
#
# This metadata is exactly what the PDF's "Re-ranking and Metadata
# Filtering" section describes: without it, the retriever has no way to
# know that the 2024 credit-score rule should take priority over an
# older, possibly outdated 2018 rule that happens to score just as high
# on pure similarity.
chunks_metadata = [
    {"text": "Applicants must be at least 21 years old.", "year": 2018, "type": "policy"},
    {"text": "Minimum monthly salary is ₹25,000.", "year": 2024, "type": "policy"},
    {"text": "Applicants must have a credit score above 650.", "year": 2024, "type": "policy"},
    {"text": "Applicants must be at least 21 years old and earn ₹25,000.", "year": 2019, "type": "policy"},
    {"text": "Minimum credit score of 700 is required.", "year": 2018, "type": "faq"},
    {"text": "Personal loans are not available to self-employed applicants.", "year": 2023, "type": "faq"},
    {"text": "EMI defaults attract a penalty of 2% per month.", "year": 2022, "type": "policy"},
]

# Pull out just the text for embedding -- the metadata (year/type) is kept
# alongside for later, but the embedding model only ever sees raw text.
chunks = [item["text"] for item in chunks_metadata]

query = "What are the age, salary, and credit score requirements for a personal loan?"
model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = model.encode(chunks)
query_embedding = model.encode([query])[0]

def vanilla_top_k(chunk_embeddings, query_embedding, k=3):
    # Same idea as before, but this version also returns the original
    # indices (top_k_idx) -- we need those to look up each chunk's
    # metadata (year/type) later for reranking.
    similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]
    top_k_idx = similarities.argsort()[-k:][::-1]
    return [chunks[i] for i in top_k_idx], top_k_idx, similarities[top_k_idx]

vanilla_results, top_k_idx, vanilla_scores = vanilla_top_k(chunk_embeddings, query_embedding, k=3)
print("\nVanilla Top-3 Results (before re-ranking):")
for text, score in zip(vanilla_results, vanilla_scores):
    print(f"  - {text}  [score: {score:.2f}]")

# For comparison, also run MMR on this same metadata-tagged set of chunks.
mmr_results = mmr(chunk_embeddings, query_embedding, k=3, lambda_param=0.7)
print("\nMMR Top-3 Results:")
for text in mmr_results:
    print(f"  - {text}")

# Let's re-rank the retrieved top-k results by 'year' (descending, newest first)
def rerank_by_year(top_k_idx, chunks_metadata):
    """
    Reorders an already-retrieved set of chunks by recency.

    Crucially, this does NOT change which chunks were retrieved -- it
    only changes the ORDER they're presented in. Vanilla top-k already
    decided the set of 3 chunks based on similarity; here we simply push
    the most recently updated policy to the front, so a generator reading
    top-to-bottom sees the current rule before any older, possibly
    superseded one.
    """
    # Extract tuples (index, year)
    idx_years = [(idx, chunks_metadata[idx]["year"]) for idx in top_k_idx]
    # Sort by year descending
    reranked = sorted(idx_years, key=lambda x: x[1], reverse=True)
    reranked_texts = [chunks_metadata[idx]["text"] for idx, _ in reranked]
    return reranked_texts

reranked_results = rerank_by_year(top_k_idx, chunks_metadata)

print("\nRe-ranked Top-3 Results (by recency):")
for text in reranked_results:
    print(f"  - {text}")